# 🦴 Medi Consult: FracAtlas YOLOv8 Bone Fracture Localization Training
This notebook trains a YOLOv8s object detector on the **FracAtlas** dataset (4,083 musculoskeletal radiographs) to predict exact bounding box coordinates of bone fractures.

- **Classification:** EfficientNet-B4 (determines if a fracture is present)
- **Localization:** YOLOv8s (locates candidate fracture sites with bounding boxes)
- **Verification:** Clinical review / Physician sign-off

In [ ]:
# Cell 1: Install required libraries
!pip install -q ultralytics opencv-python pandas scikit-learn matplotlib pyyaml

In [ ]:
# Cell 2: Download FracAtlas from official Figshare repository
import os
import requests
import zipfile
from pathlib import Path

BASE_DIR = Path("/content/fracatlas")
DOWNLOAD_DIR = BASE_DIR / "downloads"
EXTRACT_DIR = BASE_DIR / "raw"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://api.figshare.com/v2/articles/22363012"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

print("Fetching Figshare article metadata...")
response = requests.get(API_URL, headers=headers, timeout=60)
response.raise_for_status()
article = response.json()
print("Dataset:", article.get("title"))

for file_info in article.get("files", []):
    fname = file_info["name"]
    fsize = file_info["size"]
    download_url = file_info["download_url"]
    print(f"Target File: {fname} ({fsize / (1024*1024):.2f} MB)")
    
    file_path = DOWNLOAD_DIR / fname
    if not file_path.exists():
        print(f"Downloading {fname}...")
        with requests.get(download_url, headers=headers, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(file_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
        print(f"Download complete: {file_path}")
    else:
        print(f"Found existing: {file_path}")
        
    if zipfile.is_zipfile(file_path):
        print(f"Extracting {fname} to {EXTRACT_DIR}...")
        with zipfile.ZipFile(file_path, "r") as archive:
            archive.extractall(EXTRACT_DIR)
        print("Extraction complete.")

In [ ]:
# Cell 3: Inspect the extracted folders and annotation layout
from pathlib import Path

print("=== EXTRACTED FOLDER HIERARCHY ===")
for path in sorted(EXTRACT_DIR.rglob("*")):
    if path.is_dir():
        file_count = len(list(path.glob("*")))
        print(f"FOLDER: {path.relative_to(EXTRACT_DIR)} ({file_count} items)")
    elif path.suffix.lower() in [".txt", ".json", ".xml", ".csv"] and path.name in ["dataset.csv", "classes.txt", "coco.json"]:
        print(f"KEY ANNOTATION FILE: {path.relative_to(EXTRACT_DIR)}")

In [ ]:
# Cell 4: Convert and Split Dataset into Standard YOLOv8 Format (70/15/15)
import os
import shutil
import random
import yaml
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

YOLO_DATASET_DIR = Path("/content/fracatlas_yolo")
YOLO_DATASET_DIR.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    (YOLO_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Locate FracAtlas base directory inside EXTRACT_DIR
fracatlas_root = next(EXTRACT_DIR.rglob("dataset.csv")).parent
images_dir = fracatlas_root / "images"
yolo_ann_dir = fracatlas_root / "Annotations" / "YOLO"

print(f"Root FracAtlas directory: {fracatlas_root}")
print(f"Images directory: {images_dir}")
print(f"YOLO annotations directory: {yolo_ann_dir}")

# 1. Find all Fractured images (Positive cases with bounding box .txt)
fractured_dir = images_dir / "Fractured"
if not fractured_dir.exists():
    # Try alternative casing
    fractured_dir = next(images_dir.glob("*racture*"))

fractured_images = list(fractured_dir.glob("*.jpg")) + list(fractured_dir.glob("*.png"))
print(f"Found {len(fractured_images)} fractured images.")

# 2. Find Non-fractured images (Negative controls)
non_fractured_dir = images_dir / "Non_fractured"
if not non_fractured_dir.exists():
    non_fractured_dir = images_dir / "Non-fractured"
non_fractured_images = list(non_fractured_dir.glob("*.jpg")) + list(non_fractured_dir.glob("*.png"))
print(f"Found {len(non_fractured_images)} non-fractured images.")

# Sample negative controls (e.g. 1:1 or 1:2 ratio to keep localization training balanced)
random.seed(42)
sampled_negatives = random.sample(non_fractured_images, min(len(fractured_images), len(non_fractured_images)))

# Prepare paired records
records = []
for img_path in fractured_images:
    lbl_path = yolo_ann_dir / f"{img_path.stem}.txt"
    records.append({"img": img_path, "lbl": lbl_path if lbl_path.exists() else None, "is_fracture": 1})

for img_path in sampled_negatives:
    records.append({"img": img_path, "lbl": None, "is_fracture": 0})

df_records = pd.DataFrame(records)

# Stratified 70% Train, 15% Val, 15% Test split
train_df, temp_df = train_test_split(df_records, test_size=0.30, stratify=df_records["is_fracture"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["is_fracture"], random_state=42)

print(f"Split counts -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Copy files into destination
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for _, row in split_df.iterrows():
        src_img = row["img"]
        dst_img = YOLO_DATASET_DIR / "images" / split_name / src_img.name
        dst_lbl = YOLO_DATASET_DIR / "labels" / split_name / f"{src_img.stem}.txt"
        
        shutil.copy2(src_img, dst_img)
        
        if row["lbl"] and row["lbl"].exists():
            shutil.copy2(row["lbl"], dst_lbl)
        else:
            # Create empty text file for background / normal control
            dst_lbl.touch()

# Create data.yaml
yaml_content = {
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,
    "names": ["fracture"]
}

data_yaml_path = YOLO_DATASET_DIR / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f"\nCreated {data_yaml_path}:")
print(open(data_yaml_path).read())
print("Dataset preparation completed successfully!")

In [ ]:
# Cell 5: Train YOLOv8s Object Detector
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = "/content/fracatlas_yolo/data.yaml"
assert Path(DATA_YAML).exists(), "data.yaml not found"

model = YOLO("yolov8s.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    workers=2,
    project="/content/fracatlas_runs",
    name="bone_fracture_localization",
    pretrained=True,
    seed=42
)
print("Training completed.")

In [ ]:
# Cell 6: Evaluate on Test Set
best_model_path = "/content/fracatlas_runs/bone_fracture_localization/weights/best.pt"
model = YOLO(best_model_path)

metrics = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    device=0
)

print("\n--- Model Performance on Held-Out Test Set ---")
print(f"Precision (mp): {metrics.box.mp:.4f}")
print(f"Recall (mr):    {metrics.box.mr:.4f}")
print(f"mAP@50:         {metrics.box.map50:.4f}")
print(f"mAP@50-95:      {metrics.box.map:.4f}")

In [ ]:
# Cell 7: Predict and Plot on Sample Test Images
import os
import cv2
from pathlib import Path
from IPython.display import display
from PIL import Image

test_images_dir = Path("/content/fracatlas_yolo/images/test")
test_labels_dir = Path("/content/fracatlas_yolo/labels/test")

# Find valid test images with fracture labels
valid_sample_images = []
for p in test_images_dir.glob("*.jpg"):
    lbl_file = test_labels_dir / f"{p.stem}.txt"
    if p.exists() and p.stat().st_size > 5000:
        if lbl_file.exists() and lbl_file.stat().st_size > 0:
            valid_sample_images.append(p)

if not valid_sample_images:
    valid_sample_images = [p for p in test_images_dir.glob("*.jpg") if p.stat().st_size > 5000]

displayed = 0
for p in valid_sample_images:
    if displayed >= 3:
        break
    img_check = cv2.imread(str(p))
    if img_check is None:
        continue
    try:
        res = model.predict(source=str(p), conf=0.20, imgsz=640, save=False, verbose=False)
        if not res or len(res) == 0:
            continue
        plot_img = res[0].plot()
        num_boxes = len(res[0].boxes) if hasattr(res[0], 'boxes') else 0
        print(f"Image: {p.name} - Suspected Fracture Detections: {num_boxes}")
        display(Image.fromarray(plot_img[..., ::-1]))
        displayed += 1
    except Exception as e:
        print(f"Skip {p.name}: {e}")

In [ ]:
# Cell 8: Save Model Weights to Google Drive or Local Download
from google.colab import files
import shutil
import os
from pathlib import Path

best_path = Path("/content/fracatlas_runs/bone_fracture_localization/weights/best.pt")
if not best_path.exists():
    candidates = list(Path("/content").rglob("best.pt"))
    if candidates:
        best_path = candidates[0]

target_name = "bone_fracture_localization_best.pt"
shutil.copy2(str(best_path), target_name)
print(f"Downloading {target_name} ({os.path.getsize(target_name)/(1024*1024):.2f} MB) from {best_path}...")
files.download(target_name)